# ETL Pipeline — crime_incidents_messy.csv
Proyecto de ingeniería de datos: Extract → Transform → Load

Dataset real de Kaggle: 5250 filas, 33 columnas, con errores intencionales
(duplicados, nulos, categorías inconsistentes, edades/coordenadas inválidas,
fechas en formatos mixtos, montos como texto).


In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)


## 1. Extract

Sube `crime_incidents_messy.csv` a esta sesión de Colab
(ícono de carpeta en el panel izquierdo -> ícono de subir archivo).

Si prefieres traerlo directo desde Kaggle en vez de subirlo a mano:
```python
from google.colab import files
files.upload()  # selecciona tu kaggle.json (Kaggle -> Account -> Create New API Token)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d <usuario>/<nombre-del-dataset>
!unzip -o *.zip
```


In [ ]:
df_raw = pd.read_csv("crime_incidents_messy.csv")
print("Shape:", df_raw.shape)
df_raw.head()


## 2. Reporte de errores (sobre los datos crudos)

Estos conteos son equivalentes uno a uno a las fórmulas de la hoja
`Resumen_Errores` del archivo Excel — deberían darte los mismos números.


In [ ]:
def clean_key(x):
    """minúsculas + espacios colapsados + sin puntos -- para agrupar variantes de texto"""
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    s = s.replace(".", "")
    s = re.sub(r"\s+", " ", s)
    return s

reporte = {}
reporte["total_filas"] = len(df_raw)
reporte["filas_totalmente_duplicadas"] = int(df_raw.duplicated().sum())
reporte["incident_id_repetidos"] = int(df_raw["incident_id"].duplicated(keep=False).sum())

# variantes de texto (mismo criterio que COUNTIF en Excel: sensible a mayúsculas NO,
# pero sensible a espacios)
reporte["variantes_crime_type"] = int(df_raw["crime_type"].dropna().str.lower().nunique())
reporte["variantes_district"] = int(df_raw["district"].dropna().str.lower().nunique())

for col in ["suspect_age", "victim_age"]:
    vals = pd.to_numeric(df_raw[col], errors="coerce")
    reporte[f"{col}_invalidas"] = int(((vals < 0) | (vals > 100)).sum())

reporte["num_arrests_invalidos"] = int((pd.to_numeric(df_raw["num_arrests"], errors="coerce") < 0).sum())

lat = pd.to_numeric(df_raw["latitude"], errors="coerce")
reporte["latitude_invalidas"] = int(((lat < -90) | (lat > 90)).sum())

lon = pd.to_numeric(df_raw["longitude"], errors="coerce")
reporte["longitude_invalidas"] = int(((lon < -180) | (lon > 180)).sum())

reporte["fechas_no_parseadas"] = int(pd.to_datetime(df_raw["incident_datetime"], errors="coerce").isnull().sum())

print("=== Reporte de calidad de datos (crudos) ===")
for k, v in reporte.items():
    print(f"{k}: {v}")

print()
print("Nulos por columna:")
print(df_raw.isnull().sum())


## 3. Transform

### 3.1 Eliminar duplicados (por incident_id)

In [ ]:
df = df_raw.drop_duplicates(subset="incident_id", keep="first").reset_index(drop=True)
print("Shape tras quitar duplicados:", df.shape)


### 3.2 Normalizar categorías de texto (diccionarios de corrección)

In [ ]:
CRIME_TYPE_MAP = {
    "armed robbery": "Armed Robbery",
    "arson": "Arson", "arsen": "Arson", "fire setting": "Arson",
    "assault": "Assault", "asslt": "Assault", "battery": "Assault",
    "assault & battery": "Assault",
    "abduction": "Kidnapping", "kidnaping": "Kidnapping", "kidnapping": "Kidnapping",
    "b&e": "Burglary", "breaking & entering": "Burglary",
    "burglary": "Burglary", "burglry": "Burglary",
    "cyber crime": "Cyber Crime", "cybercrime": "Cyber Crime", "hacking": "Cyber Crime",
    "dui": "DUI", "d.u.i.": "DUI", "duii": "DUI", "dwi": "DUI", "drunk driving": "DUI",
    "domestic violence": "Domestic Violence", "domestc violence": "Domestic Violence",
    "dom. violence": "Domestic Violence", "dv": "Domestic Violence",
    "drug offense": "Drug Offense", "drug offence": "Drug Offense",
    "drugs": "Drug Offense", "narcotics": "Drug Offense",
    "deception": "Fraud", "fraud": "Fraud", "fraudulent activity": "Fraud",
    "online fraud": "Fraud", "scam": "Fraud",
    "graffiti": "Graffiti",
    "homicide": "Homicide", "homocide": "Homicide", "murder": "Homicide",
    "manslaughter": "Homicide",
    "larceny": "Theft", "theft/larceny": "Theft", "theft": "Theft", "stealing": "Theft",
    "property damage": "Property Damage",
    "robbery": "Robbery", "robbry": "Robbery", "roberry": "Robbery",
    "sa": "Sexual Assault", "sex assault": "Sexual Assault",
    "sexual assault": "Sexual Assault", "sexual assualt": "Sexual Assault",
    "trespass": "Trespassing", "trespassing": "Trespassing", "tresspassing": "Trespassing",
    "vandalism": "Vandalism", "vandlism": "Vandalism",
}

DISTRICT_MAP = {
    "cen": "Central", "central": "Central",
    "eas": "East", "east": "East",
    "mid": "Midtown", "midtown": "Midtown",
    "nor": "North", "north": "North",
    "northeast": "Northeast", "northwest": "Northwest",
    "sou": "South", "south": "South",
    "southeast": "Southeast", "southwest": "Southwest",
    "wes": "West", "west": "West",
}

WEAPON_MAP = {
    "bat": "Blunt Object", "blunt object": "Blunt Object",
    "firearm": "Firearm", "gun": "Firearm", "pistol": "Firearm", "rifle": "Firearm",
    "knife": "Knife",
    "hands/feet": "Unarmed", "hands": "Unarmed", "unarmed": "Unarmed",
}

SEVERITY_MAP = {
    "1": "Low", "low": "Low",
    "2": "Medium", "medium": "Medium", "med": "Medium",
    "3": "High", "high": "High",
    "4": "Critical", "critical": "Critical", "crit": "Critical",
}

CASE_STATUS_MAP = {
    "open": "Open", "closed": "Closed",
    "under investigation": "Under Investigation", "investgation": "Under Investigation",
    "pending": "Pending", "pendng": "Pending",
    "resolved": "Resolved",
}

RESOLUTION_MAP = {
    "arrest made": "Arrest Made", "arres made": "Arrest Made",
    "case dismissed": "Case Dismissed", "dismissed": "Case Dismissed",
    "no arrest": "No Arrest",
    "warning issued": "Warning Issued", "warning": "Warning Issued",
}

GENDER_MAP = {
    "f": "Female", "female": "Female",
    "m": "Male", "male": "Male",
    "other": "Other", "unknown": "Unknown",
}

BOOL_MAP = {
    "0": "No", "false": "No", "no": "No",
    "1": "Yes", "true": "Yes", "yes": "Yes",
}

def map_category(series, mapping):
    def f(x):
        key = clean_key(x)
        if key is None:
            return np.nan
        return mapping.get(key, key.title())
    return series.apply(f)

df["crime_type"] = map_category(df["crime_type"], CRIME_TYPE_MAP)
df["district"] = map_category(df["district"], DISTRICT_MAP)
df["weapon_used"] = map_category(df["weapon_used"], WEAPON_MAP)
df["severity"] = map_category(df["severity"], SEVERITY_MAP)
df["case_status"] = map_category(df["case_status"], CASE_STATUS_MAP)
df["resolution"] = map_category(df["resolution"], RESOLUTION_MAP)
df["suspect_gender"] = map_category(df["suspect_gender"], GENDER_MAP)
df["victim_gender"] = map_category(df["victim_gender"], GENDER_MAP)
df["reported_online"] = map_category(df["reported_online"], BOOL_MAP)
df["suspect_race"] = map_category(df["suspect_race"], {})
df["city"] = df["city"].str.strip().str.title()
df["state"] = df["state"].str.strip().str.upper()

df[["crime_type", "district", "weapon_used", "severity", "case_status"]].head()


### 3.3 Montos y números guardados como texto

In [ ]:
df["property_loss_usd"] = (
    df["property_loss_usd"].astype("string").str.replace(r"[\$,]", "", regex=True)
)
df["property_loss_usd"] = pd.to_numeric(df["property_loss_usd"], errors="coerce")


### 3.4 Rangos inválidos -> NaN (edades, arrestos, coordenadas)

In [ ]:
for col in ["suspect_age", "victim_age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan

df["num_arrests"] = pd.to_numeric(df["num_arrests"], errors="coerce")
df.loc[df["num_arrests"] < 0, "num_arrests"] = np.nan

df.loc[(df["latitude"] < -90) | (df["latitude"] > 90), "latitude"] = np.nan
df.loc[(df["longitude"] < -180) | (df["longitude"] > 180), "longitude"] = np.nan


### 3.5 Fechas en formatos mixtos

In [ ]:
df["incident_datetime"] = pd.to_datetime(df["incident_datetime"], errors="coerce")
print("Fechas no parseadas:", df["incident_datetime"].isnull().sum())


### 3.6 Formato de teléfono

In [ ]:
def fmt_phone(x):
    if pd.isna(x):
        return np.nan
    digits = re.sub(r"\D", "", str(x))
    if len(digits) == 10:
        return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"
    return np.nan

df["victim_phone"] = df["victim_phone"].apply(fmt_phone)


### 3.7 Rellenar nulos categóricos restantes

In [ ]:
for c in ["weapon_used", "severity", "case_status", "resolution",
          "suspect_gender", "victim_gender", "suspect_race", "reported_online"]:
    df[c] = df[c].fillna("Unknown")

print("Shape final:", df.shape)
df.isnull().sum()


## 4. Load

In [ ]:
df.to_csv("crime_data_cleaned.csv", index=False)
df.to_excel("crime_data_cleaned.xlsx", index=False)
print("Guardado. Shape final:", df.shape)


### Descargar desde Colab
```python
from google.colab import files
files.download("crime_data_cleaned.csv")
files.download("crime_data_cleaned.xlsx")
```


## 5. Salida / resumen final (ajusta según lo que pida tu profesor)

In [ ]:
print("Incidentes por distrito:")
print(df["district"].value_counts())

print("\nTipo de crimen más común:")
print(df["crime_type"].value_counts().head(10))

print("\nPérdida promedio por severidad:")
print(df.groupby("severity")["property_loss_usd"].mean().round(2))

print("\n=== Resumen de errores corregidos ===")
for k, v in reporte.items():
    print(f"{k}: {v}")
